# `Chains in Langchain`
---

### Introduction
Chains
- Sequence of Steps where the Output of one compoenent is provided to the next compoents as the Input
- Sequential Chain , Parallel Chains, Conditional Chains

# Chains in LangChain

## 1. What is a Chain?

A **Chain** in LangChain is a sequence of components connected together where the **output of one component becomes the input of the next component**.

In simple words:

> **A chain connects multiple LangChain components to perform a task as a pipeline.**

For example:

```text
User Input
    ↓
Prompt Template
    ↓
LLM
    ↓
Output Parser
    ↓
Final Output
```

Instead of manually calling each component, we can connect them into a single chain.

---

# 2. Why Do We Need Chains?

Suppose we want to build a simple AI application.

Without a chain:

```python
prompt_value = prompt.invoke({
    "topic": "RAG"
})

response = model.invoke(prompt_value)

result = parser.invoke(response)
```

We have to manually pass the output from one component to another.

With a chain:

```python
chain = prompt | model | parser
```

Then simply:

```python
result = chain.invoke({
    "topic": "RAG"
})
```

### Main Benefit

Chains make the application:

* Easier to read
* Easier to maintain
* Reusable
* Composable
* Easier to debug
* Suitable for production pipelines

---

# 3. Basic Chain Architecture

A simple LangChain application can look like:

```text
                 Input
                   ↓
          Prompt Template
                   ↓
                LLM
                   ↓
            Output Parser
                   ↓
                Output
```

For example:

```text
{
    "topic": "RAG"
}
      ↓
ChatPromptTemplate
      ↓
ChatOpenAI
      ↓
StrOutputParser
      ↓
"RAG is..."
```

---

# 4. Chains and LCEL

Modern LangChain uses **LCEL — LangChain Expression Language** to compose components.

The pipe operator:

```python
|
```

is used to connect components.

Example:

```python
chain = prompt | model | parser
```

This means:

```text
prompt
  ↓
model
  ↓
parser
```

### Important Interview Point ⭐

> **LCEL allows LangChain Runnables to be composed together using operators such as `|`, creating executable pipelines.**

---

# 5. What is a Runnable?

A **Runnable** is a component that can receive an input and produce an output.

Many LangChain components implement the Runnable interface.

For example:

* Prompt templates
* Chat models
* Output parsers
* Runnable lambdas
* Runnable maps
* Other chains

They commonly support methods such as:

```python
invoke()
batch()
stream()
```

This common interface is one of the reasons LangChain components can be easily connected.

---

# 6. Simple Chain Example

Let's create:

```text
Topic
 ↓
Prompt
 ↓
OpenAI Chat Model
 ↓
String Output
```

### Code

```python
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


model = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)


prompt = ChatPromptTemplate.from_template(
    "Explain {topic} in simple terms."
)


parser = StrOutputParser()


chain = prompt | model | parser


response = chain.invoke({
    "topic": "Vector Database"
})

print(response)
```

---

# 7. Understanding the Chain Step-by-Step

Suppose we pass:

```python
{
    "topic": "RAG"
}
```

### Step 1 — Input

```text
topic = RAG
```

### Step 2 — Prompt Template

The template:

```text
Explain {topic} in simple terms.
```

becomes:

```text
Explain RAG in simple terms.
```

### Step 3 — Chat Model

The model receives the prompt and generates an `AIMessage`.

Conceptually:

```text
AIMessage(
    content="RAG stands for Retrieval-Augmented Generation..."
)
```

### Step 4 — Output Parser

`StrOutputParser` converts the `AIMessage` into:

```text
"RAG stands for Retrieval-Augmented Generation..."
```

### Complete Flow

```text
{"topic": "RAG"}
        ↓
ChatPromptTemplate
        ↓
"Explain RAG in simple terms."
        ↓
ChatOpenAI
        ↓
AIMessage
        ↓
StrOutputParser
        ↓
String
```

---

# 8. Chain Without Output Parser

You can also create:

```python
chain = prompt | model
```

Then:

```python
response = chain.invoke({
    "topic": "RAG"
})
```

The result is an `AIMessage`.

You would typically access:

```python
response.content
```

---

# 9. Chain With Output Parser

```python
chain = prompt | model | StrOutputParser()
```

Now:

```python
response = chain.invoke({
    "topic": "RAG"
})
```

returns the text directly.

### Comparison

```text
prompt | model

Input
 ↓
Prompt
 ↓
Model
 ↓
AIMessage
```

Whereas:

```text
prompt | model | parser

Input
 ↓
Prompt
 ↓
Model
 ↓
AIMessage
 ↓
Parser
 ↓
String
```

---

# 10. Sequential Chains

A sequential chain means:

> The output of one operation is passed to another operation.

For example, imagine an application that:

1. Generates a topic
2. Creates an explanation
3. Generates interview questions

```text
Topic
 ↓
Chain 1: Generate Explanation
 ↓
Explanation
 ↓
Chain 2: Generate Interview Questions
 ↓
Questions
```

Conceptually:

```python
chain1 = prompt1 | model | parser

chain2 = prompt2 | model | parser
```

Then the output of `chain1` can become the input to `chain2`.

---

# 11. Example of Sequential Processing

```python
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI


model = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

parser = StrOutputParser()


explanation_prompt = ChatPromptTemplate.from_template(
    """
    Explain the following topic:

    {topic}
    """
)


question_prompt = ChatPromptTemplate.from_template(
    """
    Based on the following explanation,
    generate 5 interview questions:

    {explanation}
    """
)


explanation_chain = explanation_prompt | model | parser

question_chain = question_prompt | model | parser


explanation = explanation_chain.invoke({
    "topic": "RAG"
})


questions = question_chain.invoke({
    "explanation": explanation
})

print(questions)
```

### Flow

```text
"RAG"
  ↓
Explanation Chain
  ↓
Explanation
  ↓
Question Chain
  ↓
5 Interview Questions
```

---

# 12. A Better Way to Think About Chains

Think of a chain like a **factory pipeline**.

```text
Raw Material
    ↓
Machine 1
    ↓
Processed Material
    ↓
Machine 2
    ↓
Finished Product
```

In LangChain:

```text
Input
  ↓
Prompt
  ↓
LLM
  ↓
Parser
  ↓
Final Output
```

Each component performs one specific responsibility.

---

# 13. Multi-Step Chain

A real GenAI application can contain many steps:

```text
User Query
     ↓
Query Transformation
     ↓
Retriever
     ↓
Retrieved Documents
     ↓
Prompt Template
     ↓
LLM
     ↓
Output Parser
     ↓
Final Answer
```

This is much closer to how chains are used in real applications.

---

# 14. Chain for a RAG Application

A basic RAG pipeline can be thought of as:

```text
User Question
      ↓
Retriever
      ↓
Relevant Documents
      ↓
Prompt Template
      ↓
Chat Model
      ↓
Output Parser
      ↓
Answer
```

For example:

```python
rag_chain = (
    retriever
    | prompt
    | model
    | StrOutputParser()
)
```

The exact composition depends on what the retriever returns and how the prompt expects its input.

---

# 15. RunnableSequence

When you write:

```python
chain = prompt | model | parser
```

LangChain composes these runnables into a sequence.

Conceptually, this is a:

```text
RunnableSequence
```

The execution happens sequentially:

```text
Runnable 1
    ↓
Runnable 2
    ↓
Runnable 3
```

For example:

```python
chain = prompt | model | parser
```

is effectively:

```text
Runnable 1 → Prompt
Runnable 2 → Model
Runnable 3 → Parser
```

---

# 16. RunnableParallel

Sometimes we don't want sequential execution.

We may want to perform multiple operations using the same input.

For example:

```text
                Input
                  ↓
        ┌─────────┴─────────┐
        ↓                   ↓
   Generate Summary    Generate Keywords
        ↓                   ↓
        └─────────┬─────────┘
                  ↓
             Final Result
```

LangChain provides `RunnableParallel` for this kind of pattern.

Example:

```python
from langchain_core.runnables import RunnableParallel

parallel_chain = RunnableParallel(
    summary=summary_chain,
    keywords=keyword_chain
)
```

Then:

```python
result = parallel_chain.invoke({
    "text": "..."
})
```

Conceptually:

```python
{
    "summary": "...",
    "keywords": [...]
}
```

---

# 17. RunnablePassthrough

Another important LangChain concept is:

```python
RunnablePassthrough
```

It passes the input through without modifying it.

Example:

```python
from langchain_core.runnables import RunnablePassthrough
```

Conceptually:

```text
Input
  ↓
RunnablePassthrough
  ↓
Same Input
```

This becomes useful when constructing more complex chains where some values should be preserved while other values are generated.

---

# 18. RunnableLambda

`RunnableLambda` allows you to turn a Python function into a Runnable.

Example:

```python
from langchain_core.runnables import RunnableLambda


def uppercase(text):
    return text.upper()


uppercase_runnable = RunnableLambda(uppercase)
```

Then:

```python
chain = model | StrOutputParser() | uppercase_runnable
```

Flow:

```text
LLM
 ↓
String
 ↓
uppercase()
 ↓
UPPERCASE STRING
```

This allows normal Python logic to participate in LCEL pipelines.

---

# 19. Different Chain Patterns

You should understand these major patterns:

### 1. Sequential

```text
A → B → C
```

Example:

```python
prompt | model | parser
```

---

### 2. Parallel

```text
       ┌→ A ─┐
Input ─┤     ├→ Output
       └→ B ─┘
```

Example:

```python
RunnableParallel(...)
```

---

### 3. Branching

Different inputs/conditions can lead to different processing paths.

```text
             Input
               ↓
            Condition
           /         \
          /           \
      Path A         Path B
```

LangChain provides runnable branching patterns for this kind of logic.

---

### 4. Transformations

A component transforms data before passing it forward.

```text
Input
 ↓
Python Function
 ↓
Transformed Input
 ↓
LLM
```

`RunnableLambda` is useful here.

---

# 20. Chain vs Pipeline

These terms are often used similarly.

A **pipeline** describes the general idea:

```text
A → B → C
```

A **LangChain chain** is a composed set of LangChain runnables/components that performs this pipeline.

For example:

```python
prompt | model | parser
```

---

# 21. Old Chains vs Modern LCEL

This is an important interview topic.

Older LangChain tutorials often contain classes such as:

```python
LLMChain
```

For example:

```python
from langchain.chains import LLMChain
```

You may still encounter this in older codebases/tutorials.

Modern LangChain commonly favors **LCEL and Runnable composition**:

```python
chain = prompt | model | parser
```

### Interview Point ⭐

> When learning modern LangChain, focus on LCEL, Runnables, and composition rather than assuming older `Chain` abstractions are the only way to build pipelines.

---

# 22. Chain vs Agent

Another very important distinction.

## Chain

A chain follows a **predefined workflow**.

```text
A → B → C → D
```

The developer defines the sequence.

---

## Agent

An agent can dynamically decide **which action/tool to use and what to do next** based on the task.

Conceptually:

```text
User
 ↓
Agent
 ↓
Decide Action
 ↓
Tool
 ↓
Observe Result
 ↓
Decide Next Action
 ↓
Final Answer
```

### Simple Difference

> **Chain = predefined workflow**

> **Agent = dynamic decision-making workflow**

---

# 23. Practical Application — AI Study Assistant

Suppose we want an application that takes a topic and produces:

1. Explanation
2. Key points
3. Interview questions

We can design:

```text
                 Topic
                   ↓
             Prompt Template
                   ↓
                LLM
                   ↓
             Explanation
                   ↓
          ┌────────┴────────┐
          ↓                 ↓
    Key Points        Interview Questions
          ↓                 ↓
          └────────┬────────┘
                   ↓
               Final Result
```

This is a multi-step chain architecture.

---

# 24. Chain Input and Output

Every Runnable has an input and output.

For example:

```text
ChatPromptTemplate

Input:
{
    "topic": "RAG"
}

Output:
ChatPromptValue
```

Then:

```text
ChatOpenAI

Input:
ChatPromptValue

Output:
AIMessage
```

Then:

```text
StrOutputParser

Input:
AIMessage

Output:
String
```

So:

```text
Dictionary
    ↓
ChatPromptValue
    ↓
AIMessage
    ↓
String
```

This is the key concept behind LCEL.

---

# 25. Common Chain Methods

Because chains/runnables follow the Runnable interface, you will commonly encounter:

## `invoke()`

Runs the chain once.

```python
result = chain.invoke(input)
```

---

## `batch()`

Processes multiple inputs.

```python
results = chain.batch([
    {"topic": "RAG"},
    {"topic": "Embeddings"},
    {"topic": "Vector Database"}
])
```

Conceptually:

```text
Input 1 ─┐
Input 2 ─┼→ Chain → Multiple Outputs
Input 3 ─┘
```

---

## `stream()`

Streams output incrementally.

```python
for chunk in chain.stream({
    "topic": "RAG"
}):
    print(chunk, end="")
```

This is useful for chatbot UIs where you want the response to appear progressively.

---

# 26. Important Interview Questions

## Beginner

### 1. What is a Chain in LangChain?

A chain is a sequence of connected components where the output of one component is passed to the next component.

---

### 2. What is LCEL?

**LangChain Expression Language** is a declarative way to compose LangChain Runnables into pipelines.

Example:

```python
chain = prompt | model | parser
```

---

### 3. What does `|` mean in LangChain?

It composes Runnables so that the output of the left component becomes the input of the right component.

---

### 4. What is a Runnable?

A Runnable is a component that follows a common execution interface, allowing it to be invoked, streamed, batched, and composed with other Runnables.

---

## Intermediate

### 5. What happens in this chain?

```python
chain = prompt | model | StrOutputParser()
```

The flow is:

```text
Input
 ↓
Prompt Template
 ↓
Chat Model
 ↓
AIMessage
 ↓
StrOutputParser
 ↓
String
```

---

### 6. What is `RunnableSequence`?

It represents a sequence of Runnables executed one after another.

For example:

```python
prompt | model | parser
```

---

### 7. What is `RunnableParallel`?

It allows multiple Runnables to execute using the same input and combines their results.

```python
RunnableParallel(
    summary=summary_chain,
    keywords=keyword_chain
)
```

---

### 8. What is `RunnablePassthrough`?

It passes an input through unchanged and is useful when building complex runnable compositions.

---

### 9. What is `RunnableLambda`?

It wraps a normal Python function as a Runnable so that the function can participate in an LCEL chain.

---

# 27. Scenario-Based Interview Questions

### 10. You need to generate a summary and keywords from the same document. Should you use a sequential or parallel approach?

A parallel approach is appropriate because both operations can use the same document independently.

```text
              Document
              /      \
             ↓        ↓
         Summary   Keywords
```

---

### 11. You need to first summarize a document and then generate questions based on that summary. What would you use?

A sequential chain:

```text
Document
   ↓
Summary Chain
   ↓
Summary
   ↓
Question Chain
   ↓
Questions
```

---

### 12. You need to stream the generated response to a frontend. Which method would you use?

Use:

```python
chain.stream(...)
```

This allows the application to process output incrementally.

---

### 13. When would you use a Chain instead of an Agent?

Use a chain when the workflow is known and predictable.

Use an agent when the application needs dynamic decision-making, such as choosing between tools based on the task.

---

### 14. Why is LCEL useful?

LCEL provides a consistent way to compose prompts, models, parsers, retrievers, Python functions, and other Runnables into reusable pipelines.

---

# 28. Key Takeaways

* A **Chain** is a sequence of connected processing steps.
* The output of one component becomes the input of the next.
* Modern LangChain commonly uses **LCEL** to build chains.
* The pipe operator `|` connects Runnables.

```python
chain = prompt | model | parser
```

* Important Runnable concepts:

  * `RunnableSequence` → sequential execution
  * `RunnableParallel` → parallel execution
  * `RunnablePassthrough` → passes input unchanged
  * `RunnableLambda` → converts a Python function into a Runnable
* Common execution methods:

  * `invoke()` → one input
  * `batch()` → multiple inputs
  * `stream()` → incremental output
* **Chain = predefined workflow**
* **Agent = dynamic decision-making workflow**
* Older LangChain code may use classes such as `LLMChain`; modern LangChain commonly emphasizes LCEL/Runnable composition.

## ⭐ Interview One-Liner

> **A LangChain chain is a composable workflow of Runnables where each component processes the previous component's output, and LCEL provides the `|` operator and Runnable abstractions to build these pipelines cleanly.**

### Mental Model

```text
                 INPUT
                   ↓
            ┌──────────────┐
            │    Prompt    │
            └──────┬───────┘
                   ↓
            ┌──────────────┐
            │   Chat Model │
            └──────┬───────┘
                   ↓
            ┌──────────────┐
            │ OutputParser │
            └──────┬───────┘
                   ↓
                OUTPUT
```

**Remember the progression:**

```text
Prompt
  ↓
Model
  ↓
Parser
  ↓
Chain
  ↓
LCEL Composition
  ↓
Complex GenAI Application
```
